# Интро

In [ ]:
import optuna
import pandas as pd
import numpy as np
import category_encoders as ce
import matplotlib.pyplot as plt
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import cross_val_score, StratifiedKFold
from imblearn.over_sampling import SMOTE, ADASYN, BorderlineSMOTE, SVMSMOTE, RandomOverSampler
from sklearn.metrics import roc_curve, recall_score, confusion_matrix, precision_score, f1_score, accuracy_score, classification_report, roc_auc_score

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer, make_column_selector

In [3]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import src.preprocess_funsc as ppf

In [4]:
df, y = ppf.load_data(r'../datasets/train.csv')

# baseline на исходном датасете

In [5]:
a = df
n_cols = a.shape[1]

проверка на дубликаты  

In [6]:
print('количество дублирующихся строк в датасете', a.duplicated().sum())
dupl_idx = a[a.duplicated()].index
a = a.drop_duplicates()
y = y.drop(index=dupl_idx)


количество дублирующихся строк в датасете 110


In [7]:
X_train, X_test, y_train, y_test = train_test_split(a, y, stratify=y, test_size=0.2, random_state=1)

простое кодирование категориальных признаков  
и маштабирование числовых

In [8]:
cat_cols = X_train.select_dtypes(['string']).columns.to_list()
fl_cols = X_train.select_dtypes(['float']).columns.to_list()
int_cols = X_train.select_dtypes(['int']).columns.to_list()

In [9]:
cat_pipline = Pipeline([
    ('cat_imputer', SimpleImputer(strategy='most_frequent')),
    ('cat_encoder', ce.TargetEncoder()),
])

num_pipline = Pipeline([
    ('num_imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
])

prep = ColumnTransformer(
    transformers=[
        ('cat', cat_pipline, make_column_selector(dtype_include=['str'])),
        ('num', num_pipline, make_column_selector(dtype_include=['number']))
    ],
    verbose_feature_names_out=False,
).set_output(transform='pandas')

X_train_1 = prep.fit_transform(X_train, y_train)
X_test_1 = prep.transform(X_test)


## ML prediction

In [10]:
lr = LogisticRegression(random_state=1)
lr.fit(X_train_1, y_train)
y_pred = lr.predict(X_test_1)
y_proba = lr.predict_proba(X_test_1)
accuracy_sc = accuracy_score(y_test, y_pred) 
recall_sc = recall_score(y_test, y_pred)
rocauc = roc_auc_score(y_test, y_proba[:, 1])

print('Accuracy:', accuracy_sc)
print('Recall:', recall_sc)
print('ROC AUC:', rocauc)

Accuracy: 0.9163287146558522
Recall: 0.02057399723374827
ROC AUC: 0.7206533458957938


# улучшение метрик классификации

## предобработка данных 

In [11]:
X_train_2 = ppf.process_nan(X_train)
X_test_2 = ppf.process_nan(X_test)

## feature engineering

In [13]:
X_train_2 = ppf.feature_engineering(X_train_2)
X_test_2 = ppf.feature_engineering(X_test_2)
X_train.head()

,CR_PROD_CNT_IL,AMOUNT_RUB_CLO_PRC,PRC_ACCEPTS_A_EMAIL_LINK,APP_REGISTR_RGN_CODE,PRC_ACCEPTS_A_POS,PRC_ACCEPTS_A_TK,TURNOVER_DYNAMIC_IL_1M,CNT_TRAN_AUT_TENDENCY1M,SUM_TRAN_AUT_TENDENCY1M,AMOUNT_RUB_SUP_PRC,...,LDEAL_ACT_DAYS_ACC_PCT_AVG,REST_DYNAMIC_CC_3M,MED_DEBT_PRC_YWZ,LDEAL_ACT_DAYS_PCT_TR3,LDEAL_ACT_DAYS_PCT_AAVG,LDEAL_DELINQ_PER_MAXYWZ,TURNOVER_DYNAMIC_CC_3M,LDEAL_ACT_DAYS_PCT_TR,LDEAL_ACT_DAYS_PCT_TR4,LDEAL_ACT_DAYS_PCT_CURR
23391,0,0.000000,0.0,NaN,0.0,0.0,0.0,NaN,NaN,0.010405,...,0.003251,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
83034,0,0.000000,0.0,NaN,0.0,0.0,0.0,0.228571,0.323639,0.000000,...,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
277165,0,0.161037,NaN,NaN,NaN,NaN,0.0,NaN,NaN,0.300574,...,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
89170,0,0.013268,NaN,NaN,NaN,NaN,0.0,0.200000,0.255810,0.344108,...,0.230769,0.0,0.5,0.5,0.5,0.0,0.0,0.0,0.0,0.0
96271,0,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,...,NaN,0.0,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN


In [15]:
X_train_2 = prep.fit_transform(X_train_2, y_train)
X_test_2 = prep.transform(X_test_2)

## ML prediction

In [16]:
lr = LogisticRegression(random_state=1)
lr.fit(X_train_2, y_train)
y_pred = lr.predict(X_test_2)
y_proba = lr.predict_proba(X_test_2)
accuracy_sc = accuracy_score(y_test, y_pred) 
recall_sc = recall_score(y_test, y_pred)
rocauc = roc_auc_score(y_test, y_proba[:, 1])

print('Accuracy:', accuracy_sc)
print('Recall:', recall_sc)
print('ROC AUC:', rocauc)

Accuracy: 0.9180043933761406
Recall: 0.014349930843706777
ROC AUC: 0.730688069072159


In [17]:
rf = RandomForestClassifier(n_estimators=100, random_state=1)
rf.fit(X_train_2, y_train)
y_pred = rf.predict(X_test_2)
y_proba = rf.predict_proba(X_test_2)
accuracy_sc = accuracy_score(y_test, y_pred) 
recall_sc = recall_score(y_test, y_pred)
rocauc = roc_auc_score(y_test, y_proba[:, 1])

print('Accureacy:', accuracy_sc)
print('Recall:', recall_sc)
print('ROC AUC:', rocauc)

Accureacy: 0.9182296947166836
Recall: 0.03959197786998617
ROC AUC: 0.8249216831377213


In [18]:
lgbm = LGBMClassifier(n_estimators=100, random_state=1, verbose=-1)
lgbm.fit(X_train_2, y_train)
y_pred = lgbm.predict(X_test_2)
y_proba = lgbm.predict_proba(X_test_2)
accuracy_sc = accuracy_score(y_test, y_pred) 
recall_sc = recall_score(y_test, y_pred)
rocauc = roc_auc_score(y_test, y_proba[:, 1])

print('Accureacy:', accuracy_sc)
print('Recall:', recall_sc)
print('ROC AUC:', rocauc)

Accureacy: 0.9184127520558747
Recall: 0.04806362378976487
ROC AUC: 0.8402757144432842


In [19]:
cat_bst = CatBoostClassifier(iterations=100, random_seed=1, verbose=0)
cat_bst.fit(X_train_2, y_train)
y_pred = cat_bst.predict(X_test_2)
y_proba = cat_bst.predict_proba(X_test_2)
accuracy_sc = accuracy_score(y_test, y_pred) 
recall_sc = recall_score(y_test, y_pred)
rocauc = roc_auc_score(y_test, y_proba[:, 1])

print('Accureacy:', accuracy_sc)
print('Recall:', recall_sc)
print('ROC AUC:', rocauc)

Accureacy: 0.9182296947166836
Recall: 0.08419778699861688
ROC AUC: 0.8352775616113535


## Лучшая модель

выбираю лучшую модель и подбираю гиперпараметры

In [20]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [21]:
def objective(trial):
    lgmb_pars = {
        'n_estimators':  trial.suggest_int('n_estimators', 10, 1000),
        'num_leaves': trial.suggest_int('num_leaves', 10, 100),
        'learning_rate': trial.suggest_float('learning_rate', 1e-5, 0.1),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 1000)
    }
    lgbm = LGBMClassifier(**lgmb_pars)
    lgbm.fit(X_train_2, y_train)
    y_proba = lgbm.predict_proba(X_test_2)
    roc_auc = roc_auc_score(y_test, y_proba[:, 1])
    return roc_auc

In [22]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30, show_progress_bar=False)
print(study.best_params)
print(study.best_value)


{'n_estimators': 495, 'num_leaves': 51, 'learning_rate': 0.04592282533503675, 'min_child_samples': 424}
0.846818064424846
